# 一、核心问题：模型该怎么"开口说话"？

由 Torch-3- 可知分类的 y 是 0~9 的代号，没有大小远近。那模型该怎么回答"这张图是几号"？

不直接报一个号，而是报 10 个"信心值"——"我有多相信它是 T恤、它有多相信是裤子、是外套……"，然后把信心值最高的那个当答案。

问题是：模型底层是数学运算，怎么从像素算出一个"信心"？答案分三步走，Softmax 就是第三步。

# 二、‘信心值’三步走

## 第一步：算 10 个"原始分数"（logits）

线性回归输出的是一个数 y_hat = W·x + b。

分类只是把它复制成 10 份：每个类别各算一个分数:<br>
z_0 = W₀·x + b₀     （"像 T恤"的分数）<br>
z_1 = W₁·x + b₁     （"像裤子"的分数）<br>
...<br>
z_9 = W₉·x + b₉     （"像靴子"的分数）<br>

这 10 个分数合起来叫 logits（对数几率）。分数越高 = 模型越倾向该类。

拿我们那张靴子图举例，假设模型算出来的 logits 是：<br>
z = [2.0, -1.0, 0.5, -2.0, 1.0, 3.0, -0.5, 1.5, -1.5, 4.0]<br>
     T恤  裤子 套头  裙子  外套 凉鞋  衬衫 运动鞋  包   靴子
     
靴子（9号）拿 4.0 最高，凉鞋（5号）3.0 第二。看起来"差不多对了"。

## 第二步：为什么原始分数不能直接用？

这个 logits 数组不能当"信心"用，三个毛病：

1).有负数——信心值怎么能是 -2.0？
2).加起来不等于 1——10 个分数相加是乱七八糟的数；
3).没概率意义——"靴子 4.0"到底算多确信？没参照。

那能不能直接"除以总和"归一化？不行——因为分数有负数，总和可能是 0 甚至负数，归一化出来全是怪东西。

我们需要一个"加工厂"，把任何 logits 都变成：每个都≥0，加起来恰好=1。这就是 Softmax。

## 第三步：Softmax 公式（核心中的核心）

Softmax 就是一道两道工序的加工厂：

1.工序1：对每个分数取指数 eᶻ ; <br>
2.工序2：除以所有 eᶻ 的总和。<br>

公式：对第 i 个类别:
$p_i = \frac{e^{z_i}}{\sum_{j=0}^{9} e^{z_j}}$

拿上面的例子手工算一遍：
工序1 —— 取指数（把每个分数都变成正数）：
e^2.0=7.39    e^-1.0=0.37    e^0.5=1.65    e^-2.0=0.14    e^1.0=2.72  <br> e^3.0=20.09   e^-0.5=0.61    e^1.5=4.48    e^-1.5=0.22    e^4.0=54.60<br>

工序2 —— 除以总和（总和 = 7.39+0.37+1.65+0.14+2.72+20.09+0.61+4.48+0.22+54.60 ≈ 92.27）：<br>
p = [0.080, 0.004, 0.018, 0.002, 0.029, 0.218, 0.007, 0.049, 0.002, 0.592] <br> 
     T恤    裤子   套头   裙子   外套    凉鞋   衬衫   运动鞋   包  靴子<br>

检查一下：<br>
1.每个数都 ≥0 ✅;<br>
2.全部加起来 = 1 ✅（0.080+0.004+…+0.592 ≈ 1.000）;<br>
3.最大的是靴子 59.2%，凉鞋 21.8% 排第二 ✅<br>

这就是模型"开口说话"的方式：它说"我有 59.2% 的信心这是靴子，21.8% 可能是凉鞋"。


# 三、为什么要用 e 的指数？（三个理由）

1.保序：e^z 是严格单调递增的，z 越大 e^z 越大。所以排序不变——logits 里最大的，softmax 后还是最大。argmax 结果和原来一致，决策没变。

2.永不出现 0 或负数：e^z > 0 恒成立，任何 logits 进来都安全。

3.自带"放大镜"效果：e^z 是加速上升的。原本 4.0 和 3.0 只差 1，放大后 54.60 vs 20.09，差出了近 3 倍——大分数被撑得更大，小分数被压得更小。这让模型的"信心"拉得更开，决策更干脆。（这也是名字里 "soft" 的由来：它不是硬性地 0/1，而是给每个类都留一点软性的概率。）

但这也有可能会引起 e 的指数可能爆炸成大数，数值不稳的问题，后面会继续解决，如"先统一减掉最大值再算"

# 四、损失函数：交叉熵（Cross-Entropy）

有了概率输出，怎么衡量"错得有多离谱"？不能用 MSE，原因是：类别没有远近，MSE 会误导模型去"逼近"错误的数字。

交叉熵的思路极其简单粗暴：<br>
只看"真实类别"拿到的概率 p，损失 = −log(p)。<br>
模型对真实类别很有信心 → p 大 → −log(p) 小 → 损失小；<br>
模型对真实类别没信心 → p 小 → −log(p) 大 → 损失大。

拿上面的例子：真实标签是 9（靴子），模型给了 p₉ = 0.592，损失 = −log(0.592) = 0.52。损失很小，因为模型答对了且有信心。

看几个极端情况：<br>
| 真实类别概率 p | 损失 −log(p) | 含义 |
|---|---|---|
| 1.00 | 0 | 完美命中，损失 0 |
| 0.592（我们的例子） | 0.52 | 答对但有点犹豫 |
| 0.10 | 2.30 | 答对但毫无信心 |
| 0.001（答错） | 6.91 | 答错还迷之自信，重罚 |
注意最后一行——答错且自信的惩罚是最狠的。这非常合理：模型"坚定地说错"比"含糊地说错"更该挨打。

为什么叫"交叉"熵？它衡量的是"模型给的概率分布"和"真实分布"之间的交叉信息量。记住 "−log(真实类别的概率)" 这 8 个字，就抓住了交叉熵的灵魂

# 五、Softmax + 交叉熵 = 天生一对

这俩配合有个绝妙副作用：<br>
如果用 MSE 配 softmax，梯度又慢又容易卡住（softmax 的除法让梯度变得很难看）；<br>
但 softmax 输出 + 交叉熵损失，求导后梯度会化简成极其漂亮的式子：梯度 = 模型预测概率 − 真实标签（one-hot）。

通俗讲：模型预测靴子 59%，真实标签要求靴子 100%，那"误差"就是 0.59−1 = −0.41，模型就知道"靴子这路该加点分"。这一项在下一阶段代码里会变成 y_hat - y 一行，到时候会看到它的身影。

# 六、预测规则与评价指标

预测：取概率最大的那个类 = argmax(p)（因为保序，等价于取 logits 最大的那个，代码里常用后者）。<br>
评价：准确率（accuracy） = 预测对的样本数 ÷ 总样本数。不像 MSE 是连续距离，分类只看"猜没猜对"。<br>
整个过程:<br>
模型先给 10 个原始分（logits）→ Softmax 加工成 10 个加起来等于 1 的信心值（概率）→ 拿真实类别的概率算交叉熵当损失 → 梯度下降让正确类别的概率越来越高。

# 七、小结

1.logits 是所有y_hat 的合起来的值，哪个的值越大代表他的概率越大；又因为这里面的值有可能含有负数，且他们的总和不为 1 ，没有真正意义上的参照性，到底这个值代表的的概率有多大<br>

2.Softmax 的两道工序的作用是：1）对每个得到的分数取指数化，保证，每个得到的概率都大于 0 ；然后再对所有指数化后的值进行求和，并将每个值除以这个总和，得到一个处理过后的大于 0 ，小于 1 的有序概率，且这些新概率的总和为 1 ，有了参照性，这个值代表的概率有多大;<br>

3.e^z 的三个理由:<br>
1)恒正：从函数图像来理解，指数的函数图像在 X 轴的上方，值恒大于 0 ；<br>
2)保序：从函数图像来理解，指数的函数图像单调递增，具有有序性；<br>
3)放大镜：从函数图像来理解，指数的函数图像横轴越往后走，梯度或者说图像的坡度越大，那么越大的值之间的差距就越大，越小的值之间的差距会越小；<br>

4.交叉熵 为什么只看真实类别的概率？<br>
第一层（数学层）：log(0) 和 log(负数) 都是未定义的。原始 logits 里可能有 0 或负数，直接丢进 log 就爆了。<br>
第二层（语义层，更重要）：就算 logits 全是正数且 >1，−log(z) 会是负数！你想想——损失居然可以是负的？那"负损失"是什么意思？比完美还完美？<br>
整个 "往 0 压" 的意义都崩了。所以必须先进 softmax：p ∈ (0,1) → −log(p) ∈ (0, +∞)，损失只在 p=1 时才是 0，干净、有意义。
最终结论：因为总的概率加起来必须等于 1，所以把真实类别的概率往上推，另外的就被自动挤下去了。

5.为什么分类不能用 MSE 配 softmax？<br>
类别没有远近，MSE 会误导模型去"逼近"错误的数字。

6.Softmax+交叉熵的梯度会化简成 梯度 = 模型预测概率 − 真实标签（one-hot）。